<a href="https://colab.research.google.com/github/jmoussa99/football-ai/blob/main/football_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*italicized text*# Football AI

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/roboflow/sports)
[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/camera-calibration-sports-computer-vision/)
[![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://www.youtube.com/watch?v=aBVGKoNZQUw)

Let's build a Football AI system to dig deeper into match stats! We'll use computer vision and machine learning to track players, determine which team is which, and even calculate stuff like ball possession and speed. This tutorial is perfect if you want to get hands-on with sports analytics and see how AI can take your football analysis to the next level.

![football AI diagram](https://media.roboflow.com/notebooks/examples/football-ai-diagram.png)

## Before you start

### Configure your API keys

- Open your [`HuggingFace Settings`](https://huggingface.co/settings) page. Click `Access Tokens` then `New Token` to generate new token.
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store HuggingFace Access Token under the name `HF_TOKEN`.
    - Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

### Select the runtime

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Install dependencjes

**Note:** Let's install the `inference-gpu` library, which will be used to efficiently run our object detection and keypoint detection models on GPUs.

In [ ]:
!pip install -q inference

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 7.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 

**Note:** Let's install the sports repository directly from GitHub. The sports repository contains a variety of football-related utilities that we'll use along the way.

In [ ]:
!pip install -q git+https://github.com/roboflow/sports.git

  Preparing metadata (setup.py) ... done


**Note:** Let's make sure we have the latest features in the supervision library by installing version `0.23.0` or higher.

In [ ]:
!pip install -q supervision
#!pip list | grep supervision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

In [ ]:
!pip install PILLOW==10.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: PILLOW
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-doctr 1.1.0 requires Pillow>=10.1.0, but you have pillow 10.0.0 which is incompatible.
inference 1.7.2 requires pillow<13.0.0,>=12.3.0, but you have pillow 10.0.0 which is incompatible.
roboflow-workflows 0.2.3 requires pillow<13.0.0,>=12.3.0, but you have pillow 10.0.0 which is incompatible.
inference-sdk 1.7.2 requires pillow<13.0.0,>=12.3.0, but you have pillow 10.0.0 which is incompatible.
inference-models 0.39.0 requires pillow<13.0.0,>=12.3.0, but you have pillow 10.0.0 which is incompatible.
pointpats 2.5.5 requires shapely>=2.1, but you have shapely 2.0.7 which is incompa

## ball, player, goalkeeper and referee detection

In [ ]:
from inference import get_model
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
PLAYER_DETECTION_MODEL_ID = "football-players-detection-3zvbc/11"
PLAYER_DETECTION_MODEL = get_model(model_id=PLAYER_DETECTION_MODEL_ID)
FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"
FIELD_DETECTION_MODEL = get_model(model_id=FIELD_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

ModelDependencyMissing: Your `inference` configuration does not support SAM3 model. Install SAM3 dependencies and set CORE_MODEL_SAM3_ENABLED to True.
[transformers] `DepthProImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `DepthProImageProcessor` instead.
ModelDependencyMissing: Your `inference` configuration does not support YoloWorld model. Use pip install 'inference[yolo-world]' to install missing requirements.To suppress this warning, set CORE_MODEL_YOLO_WORLD_ENABLED to False.


Output()

Output()

Define which video block we want to use for the remaining cells

In [ ]:
#SOURCE_VIDEO_PATH = "/content/sample_data/alabama-vs-northwestern-new-left-side-dribble.mp4"
SOURCE_VIDEO_PATH= "/content/drive/MyDrive/Soccer_Video_Analysis/alabama-vs-northwestern-new-left-side-dribble.mp4"

In [ ]:
import supervision as sv
from tqdm import tqdm
from sports.common.team import TeamClassifier

PLAYER_ID = 2
STRIDE = 30

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
    detections = sv.Detections.from_inference(result)
    players_detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

team_classifier = TeamClassifier()
team_classifier.fit(crops)

collecting crops: 27it [08:41, 19.32s/it]


config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
team_classifier = TeamClassifier()
team_classifier.fit(crops)

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Embedding extraction: 21it [07:59, 22.82s/it]


In [ ]:
# Helper function to draw lines between each player and their two nearest teammates (same as before)
import numpy as np

def draw_all_connections_on_frame(
    frame_image: np.ndarray,
    detections: sv.Detections,
    color: sv.Color,
    line_thickness: int = 1,
    num_neighbors: int = 2
) -> np.ndarray:
    if len(detections) < 2:
        return frame_image

    output_image = frame_image.copy()
    points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER).astype(int)

    drawn_connections = set()

    for i in range(len(points)):
        current_point = points[i]
        distances = cdist([current_point], points).flatten()
        sorted_indices = np.argsort(distances)

        effective_num_neighbors = min(num_neighbors, len(points) - 1)
        if effective_num_neighbors <= 0:
            continue

        nearest_indices = sorted_indices[1:1 + effective_num_neighbors]

        for neighbor_idx in nearest_indices:
            neighbor_point = points[neighbor_idx]
            connection_key = tuple(sorted((i, neighbor_idx)))
            if connection_key not in drawn_connections:
                pt1 = tuple(current_point)
                pt2 = tuple(neighbor_point)
                cv2.line(output_image, pt1, pt2, color.as_bgr(), line_thickness, cv2.LINE_AA)
                drawn_connections.add(connection_key)
    return output_image

In [ ]:
import numpy as np
import supervision as sv

def resolve_goalkeepers_team_id(
    players: sv.Detections,
    goalkeepers: sv.Detections
) -> np.ndarray:
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)
    goalkeepers_team_id = []
    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(goalkeeper_xy - team_0_centroid)
        dist_1 = np.linalg.norm(goalkeeper_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

In [ ]:
from collections import deque
from tqdm import tqdm
import supervision as sv
import cv2
import numpy as np
from scipy.spatial.distance import cdist # Import cdist for efficient distance calculation
from sports.common.view import ViewTransformer # Import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration # Import SoccerPitchConfiguration
from sports.common.team import TeamClassifier # Import TeamClassifier

# Re-define constants (ensure these are consistent if changed elsewhere)
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3
STRIDE = 30 # Define STRIDE for crop collection

# Initialize SoccerPitchConfiguration
CONFIG = SoccerPitchConfiguration()

# Re-initialize annotators
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)
# Re-initialize LabelAnnotator, it will dynamically choose color based on class_id
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)

# Get video information
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fps = video_info.fps # Frames per second

# Define output path
OUTPUT_VIDEO_PATH = SOURCE_VIDEO_PATH[:-4]+"_output_tracking.mp4" # Reverted to original for this cell

# Create a new tracker for the full video processing
tracker = sv.ByteTrack()
tracker.reset()

# New: Dictionaries to store player positions and speeds
last_player_pixel_positions = {} # {tracker_id: (x, y)}
last_player_transformed_positions = {} # {tracker_id: (x, y)}
player_speeds_pps = {} # {tracker_id: speed_in_pixels_per_second}
player_speeds_mps = {} # {tracker_id: speed_in_meters_per_second}

# Process video frames
with sv.VideoSink(OUTPUT_VIDEO_PATH, video_info) as sink:
    for frame_idx, frame in tqdm(enumerate(sv.get_video_frames_generator(SOURCE_VIDEO_PATH)), total=video_info.total_frames):
        # ball, goalkeeper, player, referee detection
        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)

        ball_detections = detections[detections.class_id == BALL_ID]
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

        all_detections_raw = detections[detections.class_id != BALL_ID]
        all_detections_raw = all_detections_raw.with_nms(threshold=0.5, class_agnostic=True)
        all_detections = tracker.update_with_detections(detections=all_detections_raw)

        goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
        players_detections = all_detections[all_detections.class_id == PLAYER_ID]
        referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

        # team assignment
        # Note: team_classifier is assumed to be available from previous cell execution
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)

        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
            players_detections, goalkeepers_detections)

        referees_detections.class_id -= 1 # adjust class_id for referees to not conflict with team 0/1

        # Merge detections with updated class_ids for teams
        all_players_and_goalkeepers_detections = sv.Detections.merge([
            players_detections, goalkeepers_detections])

        all_detections_final = sv.Detections.merge([
            all_players_and_goalkeepers_detections, referees_detections])

        # Ensure class_id is integer type after merge for consistency
        all_detections_final.class_id = all_detections_final.class_id.astype(int)

        # --- Speed Calculation ---
        current_player_pixel_positions = {}

        # Detect pitch key points for ViewTransformer (needed for meter conversion)
        field_detection_result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        key_points = sv.KeyPoints.from_inference(field_detection_result)
        try:
          if key_points is type(None):
            pass
          if key_points.confidence is type(None):
            pass
          filter_kp = key_points.confidence[0] > 0.5
          frame_reference_points = key_points.xy[0][filter_kp]
          pitch_reference_points = np.array(CONFIG.vertices)[filter_kp]

          # Ensure there are enough points to create a transformer
          if len(frame_reference_points) >= 4 and len(pitch_reference_points) >= 4:
              transformer = ViewTransformer(
                  source=frame_reference_points,
                  target=pitch_reference_points
              )
          else:
              transformer = None # Cannot perform meter conversion

          for detection_idx in range(len(all_players_and_goalkeepers_detections)):
              tracker_id = all_players_and_goalkeepers_detections.tracker_id[detection_idx]
              current_pixel_xy = all_players_and_goalkeepers_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)[detection_idx]
              current_player_pixel_positions[tracker_id] = current_pixel_xy

              if tracker_id in last_player_pixel_positions:
                  prev_pixel_xy = last_player_pixel_positions[tracker_id]

                  # Pixel Speed
                  pixel_distance = np.linalg.norm(current_pixel_xy - prev_pixel_xy)
                  speed_pps = pixel_distance * fps
                  player_speeds_pps[tracker_id] = speed_pps

                  # Meter Speed (if transformer is available)
                  player_speeds_mps[tracker_id] = None # Indicate inability to calculate

                  if transformer:
                      current_meter_xy = transformer.transform_points(np.array([current_pixel_xy]))[0]
                      prev_meter_xy = transformer.transform_points(np.array([prev_pixel_xy]))[0]

                      if tracker_id in last_player_transformed_positions:
                        prev_meter_xy = last_player_transformed_positions[tracker_id]
                        meter_distance = np.linalg.norm(current_meter_xy - prev_meter_xy)
                        speed_mps = meter_distance * fps
                        player_speeds_mps[tracker_id] = speed_mps
                      last_player_transformed_positions[tracker_id] = current_meter_xy

              else:
                  player_speeds_pps[tracker_id] = 0.1 # No previous frame to compare
                  player_speeds_mps[tracker_id] = 0.1 # No previous frame to compare
        except:
          pass
        # Update last_player_pixel_positions for the next frame
        last_player_pixel_positions = current_player_pixel_positions

        # --- Frame Annotation ---
        labels = []
        for detection_idx in range(len(all_detections_final)):
            tracker_id = all_detections_final.tracker_id[detection_idx]
            class_name = all_detections_final.data["class_name"][detection_idx]

            # Default label
            label = f"#{tracker_id}"

            # Add speed for players and goalkeepers
            if class_name in ["player", "goalkeeper", "ball"]:
            #if class_name in ["ball"]:

                speed_pps_val = player_speeds_pps.get(tracker_id, 0.0)
                speed_mps_val = player_speeds_mps.get(tracker_id, None)

                speed_str = f"P: {speed_pps_val:.1f} px/s"
                if speed_mps_val is not None:
                    speed_str += f" | M: {speed_mps_val:.1f} m/s"

                label = f"#{tracker_id} {speed_str}"

            labels.append(label)

        annotated_frame = frame.copy()
        annotated_frame = ellipse_annotator.annotate(
            scene=annotated_frame,
            detections=all_detections_final) # Use final detections for annotation
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame,
            detections=all_detections_final,
            labels=labels)
        annotated_frame = triangle_annotator.annotate(
            scene=annotated_frame,
            detections=ball_detections)

        sink.write_frame(annotated_frame)

print(f"Processing complete. Output video saved to {OUTPUT_VIDEO_PATH}")

  0%|          | 0/475 [00:00<?, ?it/s]
Embedding extraction: 1it [00:00, 19.79it/s]
  0%|          | 1/475 [00:00<03:37,  2.18it/s]
Embedding extraction: 1it [00:00, 19.31it/s]
  0%|          | 2/475 [00:00<03:22,  2.34it/s]
Embedding extraction: 1it [00:00, 19.97it/s]
  1%|          | 3/475 [00:01<03:16,  2.40it/s]
Embedding extraction: 1it [00:00, 18.71it/s]
  1%|          | 4/475 [00:01<03:14,  2.43it/s]
Embedding extraction: 1it [00:00, 18.82it/s]
  1%|          | 5/475 [00:02<03:12,  2.44it/s]
Embedding extraction: 1it [00:00, 17.05it/s]
  1%|▏         | 6/475 [00:02<03:16,  2.38it/s]
Embedding extraction: 1it [00:00, 14.81it/s]
  1%|▏         | 7/475 [00:02<03:22,  2.31it/s]
Embedding extraction: 1it [00:00, 19.59it/s]
  2%|▏         | 8/475 [00:03<03:17,  2.36it/s]
Embedding extraction: 1it [00:00, 16.67it/s]
  2%|▏         | 9/475 [00:03<03:13,  2.40it/s]
Embedding extraction: 1it [00:00, 16.66it/s]
  2%|▏         | 10/475 [00:04<03:11,  2.42it/s]
Embedding extraction: 1it [00

Processing complete. Output video saved to /content/drive/MyDrive/Soccer_Video_Analysis/2 - Playmaking_output_tracking.mp4


In [ ]:
# --- Define and train team_classifier --- START
frame_generator_for_crops = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame_for_crop in tqdm(frame_generator_for_crops, desc='collecting crops for team classifier'):
    result = PLAYER_DETECTION_MODEL.infer(frame_for_crop, confidence=0.3)[0]
    detections = sv.Detections.from_inference(result)
    detections = detections.with_nms(threshold=0.5, class_agnostic=True)
    detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame_for_crop, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)
# --- Define and train team_classifier --- END

In [ ]:
from types import NoneType
from collections import deque
from tqdm import tqdm
import supervision as sv
import cv2
import numpy as np
from scipy.spatial.distance import cdist # Import cdist for efficient distance calculation
from sports.common.view import ViewTransformer # Import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration # Import SoccerPitchConfiguration
from sports.common.team import TeamClassifier # Import TeamClassifier

# Re-define constants (ensure these are consistent if changed elsewhere)
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3
STRIDE = 30 # Define STRIDE for crop collection

# Initialize SoccerPitchConfiguration
CONFIG = SoccerPitchConfiguration()

# Re-initialize annotators
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)
# Re-initialize LabelAnnotator, it will dynamically choose color based on class_id
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)

# Get video information
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fps = video_info.fps # Frames per second

# Define output path
OUTPUT_VIDEO_PATH = SOURCE_VIDEO_PATH[:-4]+"_output_tracking.mp4" # Reverted to original for this cell

# Create a new tracker for the full video processing
tracker = sv.ByteTrack()
tracker.reset()

# New: Dictionaries to store player positions at the start of each STRIDE interval for speed calculation
positions_at_last_stride_frame = {} # {tracker_id: (x, y)}
player_speeds_pps = {} # {tracker_id: speed_in_pixels_per_second}
player_speeds_mps = {} # {tracker_id: speed_in_meters_per_second}

# Process video frames (now processing every frame)
with sv.VideoSink(OUTPUT_VIDEO_PATH, video_info) as sink:
    for frame_idx, frame in tqdm(enumerate(sv.get_video_frames_generator(SOURCE_VIDEO_PATH)), total=video_info.total_frames):
        # ball, goalkeeper, player, referee detection
        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)

        ball_detections = detections[detections.class_id == BALL_ID]
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

        all_detections_raw = detections[detections.class_id != BALL_ID]
        all_detections_raw = all_detections_raw.with_nms(threshold=0.5, class_agnostic=True)
        all_detections = tracker.update_with_detections(detections=all_detections_raw)

        goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
        players_detections = all_detections[all_detections.class_id == PLAYER_ID]
        referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

        # team assignment
        # Note: team_classifier is assumed to be available from previous cell execution
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)

        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
            players_detections, goalkeepers_detections)

        referees_detections.class_id -= 1 # adjust class_id for referees to not conflict with team 0/1

        # Merge detections with updated class_ids for teams
        all_players_and_goalkeepers_detections = sv.Detections.merge([
            players_detections, goalkeepers_detections])

        all_detections_final = sv.Detections.merge([
            all_players_and_goalkeepers_detections, referees_detections])

        # Ensure class_id is integer type after merge for consistency
        all_detections_final.class_id = all_detections_final.class_id.astype(int)

        # --- Speed Calculation ---
        current_frame_player_pixel_positions = {} # Temporarily store current frame's positions
        current_frame_player_speed_positions = {} # Temporarily store current frame's positions

        # Detect pitch key points for ViewTransformer (needed for meter conversion)
        field_detection_result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        key_points = sv.KeyPoints.from_inference(field_detection_result)
        prev_meter_xy = {}
        try:
          filter_kp = key_points.confidence[0] > 0.5
          frame_reference_points = key_points.xy[0][filter_kp]
          pitch_reference_points = np.array(CONFIG.vertices)[filter_kp]

          # Ensure there are enough points to create a transformer
          if len(frame_reference_points) >= 4 and len(pitch_reference_points) >= 4:
              transformer = ViewTransformer(
                  source=frame_reference_points,
                  target=pitch_reference_points
              )
          else:
              transformer = None # Cannot perform meter conversion

          for detection_idx in range(len(all_players_and_goalkeepers_detections)):
              tracker_id = all_players_and_goalkeepers_detections.tracker_id[detection_idx]
              current_pixel_xy = all_players_and_goalkeepers_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)[detection_idx]
              current_frame_player_pixel_positions[tracker_id] = current_pixel_xy

              # Initialize speed for new players if they haven't been tracked before
              if tracker_id not in player_speeds_pps:
                  player_speeds_pps[tracker_id] = 0.0
                  player_speeds_mps[tracker_id] = 0.0

              # Only calculate speed if it's a 'STRIDE' frame (and not the very first frame)
              if frame_idx > 0 and frame_idx % STRIDE == 0:
                  if tracker_id in positions_at_last_stride_frame: # Check if we have a position from STRIDE frames ago
                      prev_pixel_xy = positions_at_last_stride_frame[tracker_id]

                      # Pixel Speed
                      pixel_distance = np.linalg.norm(current_pixel_xy - prev_pixel_xy)
                      time_delta = STRIDE / fps
                      speed_pps = pixel_distance / time_delta
                      player_speeds_pps[tracker_id] = speed_pps

                      # Meter Speed (if transformer is available)
                      if transformer:

                          current_meter_xy = transformer.transform_points(np.array([current_pixel_xy]))[0]
                          if tracker_id in prev_meter_xy:
                              prev_meter_xy = prev_meter_xy[tracker_id]
                              meter_distance = np.linalg.norm(current_meter_xy - prev_meter_xy)
                              speed_mps = meter_distance / time_delta
                              player_speeds_mps[tracker_id] = speed_mps
                              print(prev_meter_xy, meter_distance, speed_mps)
                          else:
                              print("no tracker_id")
                              prev_meter_xy = None
                              player_speeds_mps[tracker_id] = None
                          prev_meter_xy[tracker_id] = current_meter_xy
                      else:
                          print("no transformer")
                          player_speeds_mps[tracker_id] = None # Indicate inability to calculate
                  # If tracker_id is new since the last stride point, its speed remains 0.0 as initialized

          # After processing all detections for the current frame:
          # If it's a 'STRIDE' frame, update positions_at_last_stride_frame for the *next* calculation.
          if frame_idx % STRIDE == 0:
              positions_at_last_stride_frame = current_frame_player_pixel_positions.copy()
              if transformer:
                transformed_positions_at_last_stride_frame = current_meter_xy.copy()

        except:
          print(frame_idx)
          pass

        # --- Frame Annotation ---
        labels = []
        for detection_idx in range(len(all_detections_final)):
            tracker_id = all_detections_final.tracker_id[detection_idx]
            class_name = all_detections_final.data["class_name"][detection_idx]

            # Default label
            label = f"#{tracker_id}"

            # Add speed for players and goalkeepers
            if class_name in ["player", "goalkeeper"]:
            #if class_name in ["ball"]:

                speed_pps_val = player_speeds_pps.get(tracker_id, 0.0)
                speed_mps_val = player_speeds_mps.get(tracker_id, None)

                speed_str = f"P: {speed_pps_val:.1f} px/s"
                if speed_mps_val is not None:
                    speed_str += f" | M: {speed_mps_val:.1f} m/s"

                label = f"#{tracker_id} {speed_str}"

            labels.append(label)

        annotated_frame = frame.copy()
        annotated_frame = ellipse_annotator.annotate(
            scene=annotated_frame,
            detections=all_detections_final) # Use final detections for annotation
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame,
            detections=all_detections_final,
            labels=labels)
        annotated_frame = triangle_annotator.annotate(
            scene=annotated_frame,
            detections=ball_detections)

        # Add lines between players of Team 0 (blue)
#        team_0_players = all_detections_final[all_detections_final.class_id == 0]
#        annotated_frame = draw_all_connections_on_frame(
#            frame_image=annotated_frame,
#            detections=team_0_players,
#            color=sv.Color.from_hex('#00BFFF'), # Blue for Team 0
#            line_thickness=1,
#            num_neighbors=2
#        )

        # Add lines between players of Team 1 (pink)
#        team_1_players = all_detections_final[all_detections_final.class_id == 1]
#        annotated_frame = draw_all_connections_on_frame(
#            frame_image=annotated_frame,
#            detections=team_1_players,
#            color=sv.Color.from_hex('#FF1493'), # Pink for Team 1
#            line_thickness=1,
#            num_neighbors=2
#        )

        sink.write_frame(annotated_frame)

print(f"Processing complete. Output video saved to {OUTPUT_VIDEO_PATH}")

collecting crops for team classifier: 27it [00:12,  2.15it/s]


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Embedding extraction: 19it [00:07,  2.52it/s]
  0%|          | 0/792 [00:00<?, ?it/s]
Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:00,  4.39it/s]
  0%|          | 1/792 [00:00<09:37,  1.37it/s]

0



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:00,  5.38it/s]
  0%|          | 2/792 [00:01<08:29,  1.55it/s]

...

789



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:00,  3.47it/s]
100%|█████████▉| 791/792 [09:05<00:00,  1.43it/s]

790



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:00,  3.52it/s]
100%|██████████| 792/792 [09:06<00:00,  1.45it/s]

791
Processing complete. Output video saved to /content/drive/MyDrive/Soccer_Video_Analysis/alabama-vs-northwestern-new-left-side-dribble_output_tracking.mp4


In [ ]:
SOURCE_VIDEO_PATH